# Lesson 02 - Εξερεύνηση του Microsoft Agent Framework

Το **Microsoft Agent Framework (MAF)** είναι ένα ενοποιημένο πλαίσιο για τη δημιουργία πρακτόρων τεχνητής νοημοσύνης. Παρέχει μια καθαρή, συνθετή αρχιτεκτονική με τέσσερις βασικούς δομικούς λίθους:

- **Client** – συνδέεται με ένα endpoint μοντέλου AI και διαχειρίζεται την επικοινωνία
- **Agent** – τυλίγει έναν client με οδηγίες και ορισμούς εργαλείων
- **Tools** – επεκτείνουν τις δυνατότητες του agent με προσαρμοσμένες λειτουργίες που μπορεί να καλέσει το μοντέλο
- **Session** – διατηρεί το ιστορικό συνομιλίας για αλληλεπιδράσεις με πολλούς γύρους

Σε αυτό το μάθημα, θα δημιουργήσουμε έναν **agent κρατήσεων ταξιδιών** που ελέγχει τη διαθεσιμότητα προορισμών χρησιμοποιώντας αυτές τις έννοιες.


## Ρύθμιση


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Κατανόηση της Αρχιτεκτονικής του Πλαισίου Agent

Το Microsoft Agent Framework ακολουθεί μια αρχιτεκτονική με στρώματα:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Πελάτης** – Ένας `AzureAIProjectAgentProvider` συνδέεται με μια ανάπτυξη Azure OpenAI. Χειρίζεται τον έλεγχο ταυτότητας, τη μορφοποίηση αιτήσεων και την ανάλυση απαντήσεων.
2. **Agent** – Δημιουργείται από τον πελάτη μέσω `provider.create_agent()`, ο agent συνδυάζει την πρόσβαση στο μοντέλο με οδηγίες (system prompt) και εργαλεία.
3. **Εργαλεία** – Συναρτήσεις Python διακοσμημένες με `@tool` που μπορεί να καλεί ο agent για να εκτελεί ενέργειες ή να ανακτά δεδομένα.
4. **Συνεδρία** – Ένα αντικείμενο `AgentSession` (δημιουργημένο μέσω `agent.create_session()`) που αποθηκεύει το ιστορικό συνομιλίας, επιτρέποντας διάλογο πολλαπλών γύρων όπου ο agent θυμάται το προηγούμενο πλαίσιο.

Ας δημιουργήσουμε κάθε στρώμα βήμα προς βήμα.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Προσθήκη Εργαλείων με το Διακοσμητή @tool

Τα εργαλεία επιτρέπουν στους πράκτορες να λαμβάνουν δράσεις πέρα από την παραγωγή κειμένου. Ο διακοσμητής `@tool` μετατρέπει μια κανονική συνάρτηση Python σε κάτι που μπορεί να καλέσει ο πράκτορας.

Σημαντικά σημεία:
- Χρησιμοποιήστε `Annotated[type, "περιγραφή"]` ώστε το μοντέλο να κατανοεί κάθε παράμετρο.
- Το docstring γίνεται η περιγραφή του εργαλείου που βλέπει το μοντέλο.
- Το `approval_mode="never_require"` σημαίνει ότι το εργαλείο εκτελείται αυτόματα χωρίς επιβεβαίωση χρήστη.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Δημιουργία ενός Πράκτορα με Εργαλεία

Τώρα συνδυάζουμε τον πελάτη, τις οδηγίες και τα εργαλεία σε έναν πράκτορα. Οι `instructions` λειτουργούν ως το σύστημα εντολών — καθορίζουν την προσωπικότητα και τη συμπεριφορά του πράκτορα.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Συνομιλίες Πολλαπλών Γύρων με Συνεδρίες

Μια `AgentSession` (δημιουργείται μέσω `agent.create_session()`) παρακολουθεί όλα τα μηνύματα σε μια συνομιλία. Με τη μετάδοση της ίδιας συνεδρίας σε κάθε κλήση `agent.run()`, ο παράγοντας έχει πρόσβαση στο πλήρες ιστορικό της συνομιλίας και μπορεί να ανατρέξει σε προηγούμενα μηνύματα.

Περνάμε `tools=[check_destination_availability]` ώστε ο παράγοντας να μπορεί να καλεί τον ελεγκτή διαθεσιμότητας μας σε κάθε γύρο.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Περίληψη

Σε αυτό το μάθημα εξερευνήσατε τους τέσσερις πυλώνες του Microsoft Agent Framework:

| Έννοια | Τι Μάθατε |
|---------|------------------|
| **Πελάτης** | Το `AzureAIProjectAgentProvider` συνδέεται με το Azure OpenAI με πιστοποίηση βάσει διαπιστευτηρίων |
| **Πράκτορας** | Το `provider.create_agent()` δημιουργεί μια σύνδεση μοντέλου με οδηγίες και όνομα |
| **Εργαλεία** | Ο διακοσμητής `@tool` εκθέτει συναρτήσεις Python για να καλεί ο πράκτορας |
| **Συνεδρία** | Το `agent.create_session()` διατηρεί το ιστορικό συνομιλίας σε πολλαπλές στροφές |

Αυτά τα δομικά στοιχεία συνθέτουν μαζί πράκτορες που μπορούν να διεξάγουν φυσικές συνομιλίες, να καλούν εξωτερικές λειτουργίες και να διατηρούν το πλαίσιο — το θεμέλιο για πιο προηγμένα πρότυπα πράκτορα σε επόμενα μαθήματα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση Ευθυνών**:  
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία αυτόματης μετάφρασης AI [Co-op Translator](https://github.com/Azure/co-op-translator). Παρότι προσπαθούμε για ακρίβεια, παρακαλούμε να λάβετε υπόψη ότι οι αυτόματες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρανοήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
